In [6]:
import tensorflow as tf
print(tf.__version__) #checking tensorflow version
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus)) # check if the GPU is available

if gpus:
    print("GPU Details:", gpus)
else:
    print("No GPU detected. TensorFlow is running on CPU.")

2.20.0
Num GPUs Available:  1
GPU Details: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
""""external libaries import configuration"""
import tensorflow as tf
from tensorflow import keras # type: ignore
from tensorflow.keras import layers  # type: ignore

import numpy as np
import matplotlib.pyplot as plt

import os
import pathlib

In [ ]:
BASE_DIR = "/mnt/d/US_Banknote_Classification/US_BANK_NOTE_CLASSIFICATION/Data"
DATA_DIR = f"{BASE_DIR}/USA currency"
TEST_DIR = f"{BASE_DIR}/Test_Set"

img_size = (160, 160)
batch_size = 32
SEED = 123
VAL_SPLIT = 0.1

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

class_names = train_ds.class_names

# Test_Set mirrors the training class folders, so the same class_names are passed
# explicitly to keep the integer labels aligned across all three datasets.
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int',
    class_names=class_names,
    shuffle=False
)

IMAGE_EXTS = ('.bmp', '.gif', '.jpeg', '.jpg', '.png')

def count_images(directory):
    """Count the image files a loader would pick up under a directory."""
    return sum(1 for p in pathlib.Path(directory).rglob('*')
               if p.suffix.lower() in IMAGE_EXTS)

# Counts come from the directories rather than from the dataset objects: Keras
# attaches .file_paths to whatever it returns, and any later .map/.cache/.prefetch
# hands back a new object without it.
n_total = count_images(DATA_DIR)
n_val = int(VAL_SPLIT * n_total)
n_train = n_total - n_val
n_test = count_images(TEST_DIR)

# The split is a prefix/suffix slice of one shuffled list, so train and val are
# disjoint by construction as long as the two parts still account for every file.
assert n_train + n_val == n_total, f"split does not cover the data: {n_train} + {n_val} != {n_total}"
print(f"train: {n_train}  val: {n_val}  test: {n_test}")

print("class_names:", class_names)
